# 52 - Representacao hibrida TF-IDF + MPNet

Este notebook implementa uma representacao hibrida para cada descricao normalizada: concatena o vetor lexical TF-IDF e o embedding semantico MPNet, com pesos ajustados por `ALPHA`. A funcao de combinacao e independente de qualquer algoritmo de clusterizacao.

**Pergunta:** como construir uma matriz que preserve sinais lexicais e semanticos, com as duas partes normalizadas e registros alinhados?

Execute todas as celulas em ordem, em uma sessao limpa. A execucao usa uma amostra fixa de ate 5.000 descricoes para limitar o custo de gerar embeddings. Nenhum arquivo de dados ou modelo e gravado.

## 1. Configuracoes

`ALPHA` controla a contribuicao das representacoes:

- `1.0`: somente TF-IDF;
- `0.0`: somente MPNet;
- `0.5`: contribuicoes iguais;
- valores intermediarios: os blocos sao multiplicados por `sqrt(alpha)` e `sqrt(1 - alpha)`. Como ocupam colunas distintas, contribuem para a norma ao quadrado nas proporcoes `alpha` e `1 - alpha`.

Altere `ALPHA` abaixo para configurar o experimento. As configuracoes de TF-IDF, modelo, amostra e lote tambem ficam reunidas aqui.

In [5]:
from datetime import datetime, timezone
import html
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

# Localiza a raiz para funcionar com Jupyter iniciado na raiz ou em notebooks/.
PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Nao encontrei a raiz do repositorio com pyproject.toml.")
DATASET_PATH = (
    PROJECT_ROOT / "data" / "processed" / "2025_NCMs_filtrados-sem-duplicatas"
    / "2025_NCMs_filtrados-sem-duplicatas-202501-202512.parquet"
)
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
MAX_DESCRIPTIONS = 5_000  # None usa todas as descricoes.
SAMPLE_RANDOM_STATE = 42
ENCODE_BATCH_SIZE = 64

TFIDF_ANALYZER = "word"
TFIDF_NGRAM_RANGE = (1, 2)
TFIDF_MIN_DF = 5
TFIDF_MAX_FEATURES = 30_000
ALPHA = 0.5
OUTPUT_ROOT = PROJECT_ROOT / "reports" / "tfidf_mpnet_hybrid"
PERIOD_LABEL = "202501-202512"
SVD_COMPONENTS = 50
TSNE_PERPLEXITY = 30
TSNE_MAX_ITER = 1_000
TSNE_RANDOM_STATE = 42


## 2. Funcao de combinacao

A funcao valida `alpha`, dimensoes e identificadores dos registros. Ela normaliza cada linha em L2 antes de aplicar os pesos. Para `alpha` entre 0 e 1, a matriz TF-IDF permanece esparsa: embeddings densos sao convertidos para CSR para que a concatenacao resultante tambem seja CSR. A matriz TF-IDF nunca e convertida em array denso.

Nos extremos, a funcao retorna somente a representacao ativa e evita construir um bloco zerado. Para pesos intermediarios, a matriz combinada armazena o bloco MPNet, normalmente denso; confira a memoria antes de aumentar muito a amostra.

In [6]:
def combine_representations(tfidf_matrix, mpnet_matrix, alpha, tfidf_ids, mpnet_ids):
    """Normalize e concatena TF-IDF e MPNet com pesos controlados por alpha.

    Os identificadores explicitam a ordem das linhas e precisam coincidir.
    Para 0 < alpha < 1, retorna CSR quando TF-IDF e esparso; nos extremos,
    retorna somente a representacao ativa no formato esparso ou denso de entrada.
    """
    try:
        alpha = float(alpha)
    except (TypeError, ValueError) as exc:
        raise ValueError("alpha deve ser um numero finito no intervalo [0, 1].") from exc
    if not np.isfinite(alpha) or not 0.0 <= alpha <= 1.0:
        raise ValueError("alpha deve ser um numero finito no intervalo [0, 1].")
    if tfidf_matrix.ndim != 2 or mpnet_matrix.ndim != 2:
        raise ValueError("As duas representacoes devem ser matrizes bidimensionais.")
    if tfidf_matrix.shape[0] != mpnet_matrix.shape[0]:
        raise ValueError("As representacoes devem ter a mesma quantidade de linhas.")

    tfidf_ids, mpnet_ids = np.asarray(tfidf_ids), np.asarray(mpnet_ids)
    rows = tfidf_matrix.shape[0]
    if len(tfidf_ids) != rows or len(mpnet_ids) != rows:
        raise ValueError("Cada matriz deve ter um identificador por linha.")
    if len(np.unique(tfidf_ids)) != rows or len(np.unique(mpnet_ids)) != rows:
        raise ValueError("Os identificadores de linha devem ser unicos em cada representacao.")
    if not np.array_equal(tfidf_ids, mpnet_ids):
        raise ValueError("A ordem dos registros difere entre TF-IDF e MPNet.")

    if alpha == 1.0:
        return normalize(tfidf_matrix, norm="l2", axis=1, copy=True)
    if alpha == 0.0:
        return normalize(mpnet_matrix, norm="l2", axis=1, copy=True)

    tfidf_normalized = normalize(tfidf_matrix, norm="l2", axis=1, copy=True)
    mpnet_normalized = normalize(mpnet_matrix, norm="l2", axis=1, copy=True)
    tfidf_weighted = tfidf_normalized * np.sqrt(alpha)
    mpnet_weighted = mpnet_normalized * np.sqrt(1.0 - alpha)

    if sparse.issparse(tfidf_weighted):
        # Mantem TF-IDF esparso, sem converte-lo em array denso.
        return sparse.hstack(
            (tfidf_weighted, sparse.csr_matrix(mpnet_weighted)), format="csr"
        )
    return np.concatenate((tfidf_weighted, mpnet_weighted), axis=1)


def project_hybrid_with_tsne(matrix, max_components, perplexity, random_state):
    """Reduz a matriz hibrida com SVD e calcula coordenadas t-SNE 3D."""
    if matrix.shape[0] < 3:
        raise ValueError("O t-SNE requer pelo menos tres descricoes.")
    if matrix.shape[1] < 2:
        raise ValueError("A matriz hibrida precisa ter ao menos duas features.")
    component_count = min(max_components, matrix.shape[1] - 1)
    reduced = TruncatedSVD(n_components=component_count, random_state=random_state).fit_transform(matrix)
    effective_perplexity = min(perplexity, matrix.shape[0] - 1)
    coordinates = TSNE(
        n_components=3,
        perplexity=effective_perplexity,
        max_iter=TSNE_MAX_ITER,
        init="random",
        learning_rate="auto",
        random_state=random_state,
    ).fit_transform(reduced)
    return coordinates, component_count, effective_perplexity


def save_interactive_tsne_html(coordinates, items, title, output_path):
    """Salva um HTML local com visualizacao 3D rotacionavel e tooltip."""
    payload = {
        "title": title,
        "points": [
            {"x": float(point[0]), "y": float(point[1]), "z": float(point[2]),
             "description": str(description), "frequency": int(frequency)}
            for point, description, frequency in zip(
                coordinates, items["descricao_normalizada"], items["frequencia"]
            )
        ],
    }
    payload_json = json.dumps(payload, ensure_ascii=False).replace("<", "\\u003c")
    document = r"""<!doctype html>
<html lang="pt-BR"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>__TITLE__</title><style>
body{margin:0;color:#202124;font:14px system-ui,sans-serif}header{display:flex;align-items:center;gap:14px;padding:12px 18px}h1{margin:0;font-size:18px}#summary{color:#555}.help{padding:0 18px 8px;color:#555}canvas{display:block;width:100vw;height:calc(100vh - 80px);cursor:grab}canvas:active{cursor:grabbing}#tooltip{position:fixed;display:none;max-width:360px;padding:8px 10px;color:white;background:#222e;border-radius:5px;pointer-events:none}
</style></head><body><header><h1 id="title"></h1><span id="summary"></span></header>
<div class="help">Arraste para girar; use a roda do mouse para aproximar. Passe sobre um ponto para ver descricao e frequencia.</div>
<canvas id="chart"></canvas><div id="tooltip"></div><script id="chart-data" type="application/json">__PAYLOAD__</script><script>
const data=JSON.parse(document.getElementById('chart-data').textContent),canvas=document.getElementById('chart'),ctx=canvas.getContext('2d'),tip=document.getElementById('tooltip');
document.getElementById('title').textContent=data.title;document.getElementById('summary').textContent=`${data.points.length.toLocaleString()} descricoes; t-SNE 3D da matriz hibrida`;
let yaw=.55,pitch=.35,zoom=1,drag=null,projected=[];
function resize(){const r=devicePixelRatio||1;canvas.width=Math.round(canvas.clientWidth*r);canvas.height=Math.round(canvas.clientHeight*r);ctx.setTransform(r,0,0,r,0,0);draw()}
function draw(){const w=canvas.clientWidth,h=canvas.clientHeight,p=data.points;ctx.clearRect(0,0,w,h);if(!p.length)return;const cx=p.reduce((s,q)=>s+q.x,0)/p.length,cy=p.reduce((s,q)=>s+q.y,0)/p.length,cz=p.reduce((s,q)=>s+q.z,0)/p.length,rad=Math.max(...p.map(q=>Math.hypot(q.x-cx,q.y-cy,q.z-cz)))||1,scale=Math.min(w,h)*.39*zoom,ca=Math.cos(yaw),sa=Math.sin(yaw),cp=Math.cos(pitch),sp=Math.sin(pitch);
projected=p.map((q,i)=>{const x=(q.x-cx)/rad,y=(q.y-cy)/rad,z=(q.z-cz)/rad,rx=x*ca-z*sa,rz=x*sa+z*ca,ry=y*cp-rz*sp,depth=y*sp+rz*cp,perspective=2.8/(2.8-depth*.55);return{x:w/2+rx*scale*perspective,y:h/2-ry*scale*perspective,depth,index:i}}).sort((a,b)=>a.depth-b.depth);
for(const q of projected){ctx.globalAlpha=.78;ctx.fillStyle='#3366cc';ctx.beginPath();ctx.arc(q.x,q.y,3,0,Math.PI*2);ctx.fill()}ctx.globalAlpha=1}
canvas.addEventListener('pointerdown',e=>{drag=[e.clientX,e.clientY];canvas.setPointerCapture(e.pointerId)});
canvas.addEventListener('pointermove',e=>{if(drag){yaw+=(e.clientX-drag[0])*.008;pitch=Math.max(-1.45,Math.min(1.45,pitch+(e.clientY-drag[1])*.008));drag=[e.clientX,e.clientY];draw();return}const r=canvas.getBoundingClientRect(),x=e.clientX-r.left,y=e.clientY-r.top;let best=null,dist=100;for(const q of projected){const d=Math.hypot(q.x-x,q.y-y);if(d<dist){dist=d;best=q}}if(best&&dist<10){const q=data.points[best.index];tip.textContent=`${q.description}: frequencia: ${q.frequency}`;tip.style.display='block';tip.style.left=`${e.clientX+12}px`;tip.style.top=`${e.clientY+12}px`}else tip.style.display='none'});
canvas.addEventListener('pointerup',()=>drag=null);canvas.addEventListener('wheel',e=>{e.preventDefault();zoom=Math.max(.3,Math.min(5,zoom*(e.deltaY<0?1.1:.9)));draw()},{passive:false});window.addEventListener('resize',resize);resize();
</script></body></html>"""
    document = document.replace("__TITLE__", html.escape(title)).replace("__PAYLOAD__", payload_json)
    output_path.write_text(document, encoding="utf-8")


## 3. Carregar e ordenar descricoes

A unidade de analise e uma descricao normalizada unica. As linhas sao agrupadas e ordenadas antes da amostragem; `descricao_id` identifica cada linha e sera passado com as duas representacoes para validar a ordem.

In [7]:
if not DATASET_PATH.is_file():
    raise FileNotFoundError(f"Arquivo Parquet nao encontrado: {DATASET_PATH}")
items = pd.read_parquet(DATASET_PATH)
required_columns = {"descricao_normalizada", "frequencia"}
missing_columns = required_columns.difference(items.columns)
if missing_columns:
    raise ValueError(f"Colunas obrigatorias ausentes: {sorted(missing_columns)}")
items = items[["descricao_normalizada", "frequencia"]].copy()
items["descricao_normalizada"] = items["descricao_normalizada"].astype("string").str.strip()
items["frequencia"] = pd.to_numeric(items["frequencia"], errors="raise")
items = items.dropna(subset=["descricao_normalizada", "frequencia"])
items = items.loc[items["descricao_normalizada"].ne("") & items["frequencia"].ge(1)]
items = (items.groupby("descricao_normalizada", as_index=False, sort=False)["frequencia"]
         .sum().sort_values("descricao_normalizada").reset_index(drop=True))
items["frequencia"] = items["frequencia"].astype("int64")
if MAX_DESCRIPTIONS is not None and len(items) > MAX_DESCRIPTIONS:
    items = (items.sample(n=MAX_DESCRIPTIONS, random_state=SAMPLE_RANDOM_STATE)
             .sort_values("descricao_normalizada").reset_index(drop=True))
items.insert(0, "descricao_id", np.arange(len(items), dtype=np.int64))
if items.empty:
    raise ValueError("Nao ha descricoes validas para representar.")
display(items.head())
print(f"Descricoes representadas: {len(items):,}")


,descricao_id,descricao_normalizada,frequencia
0,0,00007 detergente composicao tensoativo anionic...,1
1,1,00054 bolo alimenticio sabor maracuja tipo tor...,2
2,2,00099 salgados diversos tipo rissole tamanho p...,1
3,3,001 126 pilha tam peq aa 1 5v alcalina,1
4,4,001 229 bateria 3v cr2032,1


Descricoes representadas: 5,000


## 4. Gerar e combinar as representacoes

As duas matrizes sao criadas a partir da mesma lista ordenada de descricoes. O codificador retorna embeddings sem normalizacao para que a funcao aplique explicitamente a mesma regra L2 aos dois blocos.

In [8]:
texts = items["descricao_normalizada"].astype(str).tolist()
row_ids = items["descricao_id"].to_numpy()
vectorizer = TfidfVectorizer(
    analyzer=TFIDF_ANALYZER, ngram_range=TFIDF_NGRAM_RANGE,
    min_df=TFIDF_MIN_DF, max_features=TFIDF_MAX_FEATURES,
    sublinear_tf=True, norm=None,
)
tfidf_matrix = vectorizer.fit_transform(texts)
encoder = SentenceTransformer(MODEL_NAME)
mpnet_matrix = encoder.encode(
    texts, batch_size=ENCODE_BATCH_SIZE, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=False,
).astype(np.float32, copy=False)
if tfidf_matrix.shape[0] != len(items) or mpnet_matrix.shape[0] != len(items):
    raise ValueError("Uma representacao nao preservou uma linha por descricao.")
hybrid_matrix = combine_representations(
    tfidf_matrix, mpnet_matrix, ALPHA, row_ids, row_ids.copy()
)
if sparse.issparse(hybrid_matrix):
    row_norms = np.sqrt(np.asarray(hybrid_matrix.multiply(hybrid_matrix).sum(axis=1)).ravel())
    stored_values, matrix_format = hybrid_matrix.nnz, hybrid_matrix.getformat()
else:
    row_norms = np.linalg.norm(hybrid_matrix, axis=1)
    stored_values, matrix_format = int(np.count_nonzero(hybrid_matrix)), "dense"
summary = pd.DataFrame([{
    "alpha": ALPHA, "descricoes": len(items),
    "tfidf_features": tfidf_matrix.shape[1],
    "mpnet_dimensions": mpnet_matrix.shape[1],
    "hybrid_dimensions": hybrid_matrix.shape[1],
    "formato_hibrido": matrix_format, "valores_armazenados": stored_values,
    "norma_minima": float(row_norms.min()), "norma_maxima": float(row_norms.max()),
}])
display(summary)
print("A matriz hibrida esta pronta para ser usada por um algoritmo de sua escolha.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

,alpha,descricoes,tfidf_features,mpnet_dimensions,hybrid_dimensions,formato_hibrido,valores_armazenados,norma_minima,norma_maxima
0,0.5,5000,1419,768,2187,csr,3863271,0.707107,1.0


A matriz hibrida esta pronta para ser usada por um algoritmo de sua escolha.


## 5. Projetar e salvar o HTML 3D

A matriz hibrida e reduzida por TruncatedSVD antes do t-SNE, que gera as coordenadas 3D. O HTML interativo mostra as descricoes como pontos de uma unica camada, pois este notebook nao executa clusterizacao. Cada execucao cria uma pasta nova em `reports/tfidf_mpnet_hybrid/` e salva o HTML sem sobrescrever resultados anteriores.

In [9]:
run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_dir = OUTPUT_ROOT / f"{PERIOD_LABEL}_{run_timestamp}"
output_dir.mkdir(parents=True, exist_ok=False)
coordinates, svd_dimensions, effective_perplexity = project_hybrid_with_tsne(
    matrix=hybrid_matrix,
    max_components=SVD_COMPONENTS,
    perplexity=TSNE_PERPLEXITY,
    random_state=TSNE_RANDOM_STATE,
)
html_path = output_dir / "representacao_hibrida_tsne_3d.html"
save_interactive_tsne_html(
    coordinates=coordinates,
    items=items,
    title=f"t-SNE 3D - TF-IDF + MPNet (alpha={ALPHA:g})",
    output_path=html_path,
)
print(f"SVD: {svd_dimensions} dimensoes; perplexidade t-SNE: {effective_perplexity}")
print(f"HTML interativo salvo em: {html_path.relative_to(PROJECT_ROOT)}")


SVD: 50 dimensoes; perplexidade t-SNE: 30
HTML interativo salvo em: reports\tfidf_mpnet_hybrid\202501-202512_20261017T193102838064Z\representacao_hibrida_tsne_3d.html


## 6. Interpretacao e limites

- `ALPHA` maior destaca correspondencias lexicais; menor destaca proximidade semantica MPNet. O peso altera a contribuicao de cada bloco na geometria da matriz concatenada.
- Para `0 < ALPHA < 1`, cada bloco e normalizado em L2 antes de receber seu peso. A norma total da linha e 1 quando ambos os vetores sao nao nulos.
- A saida mista e CSR, preservando TF-IDF esparso sem densifica-lo. Como MPNet normalmente produz embeddings densos, sua conversao para CSR pode ocupar memoria adicional.
- Descricoes sem termos sobreviventes ao vocabulario TF-IDF ficam com bloco lexical zerado; MPNet continua representando a descricao.
- t-SNE e uma projecao exploratoria: distancias no HTML nao substituem as distancias na matriz original nem indicam clusters validados.
- A funcao de combinacao nao ajusta nem depende de clusterizacao. `hybrid_matrix` mantem a ordem de `items` e pode ser passada a algoritmos independentes.